In [11]:
import sqlite3
conn = sqlite3.connect('hospital_compare.db')


In [12]:
cursor = conn.cursor()

In [13]:
cursor.execute("""
SELECT name 
FROM sqlite_master 
WHERE type='table';
""")

tables = cursor.fetchall()

tables

[('hospitals',),
 ('measures',),
 ('hospital_measures',),
 ('hac_payment_reduction',)]

In [14]:
pip install pandas

Note: you may need to restart the kernel to use updated packages.


In [15]:
import pandas as pd

df = pd.read_sql_query("SELECT * FROM hospitals", conn)

df.head()

,facility_id,hospital_name,address,city,state,zip_code,county,phone_number,hospital_type,hospital_ownership,emergency_services,beds,lat,lon,in_2011_data,in_2026_data,has_name_conflict_2026
0,010001,SOUTHEAST HEALTH MEDICAL CENTER,1108 ROSS CLARK CIRCLE,DOTHAN,AL,36301,HOUSTON,(334) 793-8701,Acute Care Hospitals,Government - Hospital District or Authority,1.0,None,None,None,1,1,0
1,010005,MARSHALL MEDICAL CENTERS,2505 U S HIGHWAY 431 NORTH,BOAZ,AL,35957,MARSHALL,(256) 593-8310,Acute Care Hospitals,Government - Hospital District or Authority,1.0,None,None,None,1,1,0
2,010006,NORTH ALABAMA MEDICAL CENTER,1701 VETERANS DRIVE,FLORENCE,AL,35630,LAUDERDALE,(256) 768-8400,Acute Care Hospitals,Government - Hospital District or Authority,1.0,None,None,None,1,1,0
3,010007,MIZELL MEMORIAL HOSPITAL,702 N MAIN ST,OPP,AL,36467,COVINGTON,(334) 493-3541,Acute Care Hospitals,Voluntary Non-Profit - Private,1.0,None,None,None,1,1,0
4,010008,CRENSHAW COMMUNITY HOSPITAL,101 HOSPITAL CIRCLE,LUVERNE,AL,36049,CRENSHAW,(334) 335-3374,Acute Care Hospitals,Proprietary,1.0,None,None,None,1,1,0


In [16]:
import pandas as pd

df = pd.read_sql_query("SELECT * FROM measures", conn)

df.head()

,measure_id,measure_name,topic,file_year
0,LEGACY2011_MORTALITY_HEART_ATTACK,Mortality Rate - Heart Attack (2011),Legacy_2011,2011
1,LEGACY2011_MORTALITY_HEART_FAILURE,Mortality Rate - Heart Failure (2011),Legacy_2011,2011
2,LEGACY2011_MORTALITY_PNEUMONIA,Mortality Rate - Pneumonia (2011),Legacy_2011,2011
3,LEGACY2011_READMISSION_HEART_ATTACK,Readmission Rate - Heart Attack (2011),Legacy_2011,2011
4,LEGACY2011_READMISSION_HEART_FAILURE,Readmission Rate - Heart Failure (2011),Legacy_2011,2011


In [17]:
import pandas as pd

df = pd.read_sql_query("SELECT * FROM hospital_measures", conn)

df.head()

,facility_id,measure_id,file_year,value
0,010001,LEGACY2011_MORTALITY_HEART_ATTACK,2011,14.3
1,010005,LEGACY2011_MORTALITY_HEART_ATTACK,2011,18.5
2,010006,LEGACY2011_MORTALITY_HEART_ATTACK,2011,18.1
3,010007,LEGACY2011_MORTALITY_HEART_ATTACK,2011,NaN
4,010008,LEGACY2011_MORTALITY_HEART_ATTACK,2011,NaN


In [18]:
import pandas as pd

df = pd.read_sql_query("SELECT * FROM hac_payment_reduction", conn)

df.head()

,facility_id,file_year,payment_reduction
0,010001,2026,No
1,010005,2026,Yes
2,010006,2026,No
3,010007,2026,Yes
4,010008,2026,No


In [19]:
"""============================================================================
1. Rank U.S. states by average AMI (heart attack)
============================================================================"""
with open('sql/queries/01_state_mortality_rankings_2022.sql', 'r') as file:
    query = file.read()

df = pd.read_sql_query(query, conn)

df.head(5)

,state,avg_mortality_rate,hospitals_reporting,national_avg,diff_vs_national,worst_to_best_rank


In [20]:
"""=================================================================================
2. Compare the AMI (heart attack) mortality rate of hospitals across the 11-year gap.
===================================================================================="""
with open('sql/queries/02_then_vs_now_2011_2022.sql', 'r') as file:
    query = file.read()

df = pd.read_sql_query(query, conn)

df.head(5)

,hospital_name,state,mortality_rate_2011,mortality_rate_2022,change_in_rate


In [21]:
"""============================================================================
3. Hospitals that rank in the WORST national quartile on BOTH MRSA and AMI
============================================================================"""
with open('sql/queries/03_cross_topic_worst_quartile.sql', 'r') as file:
    query = file.read()

df = pd.read_sql_query(query, conn)

df.head(5)

,hospital_name,state,mrsa_sir,ami_mortality_rate


In [26]:
"""================================================================================
4. Mortality outcomes across all 4 Complications & Deaths conditions at once from 2022
================================================================================"""
with open('sql/queries/04_ownership_vs_outcomes.sql', 'r') as file:
    query = file.read()

df = pd.read_sql_query(query, conn)

df.head(5)

,hospital_ownership,num_hospitals,avg_ami_mortality,avg_hf_mortality,avg_copd_mortality,avg_stroke_mortality
0,Voluntary Non-Profit - Private,1488,None,None,34.89,57.74
1,Voluntary Non-Profit - Other,569,None,None,31.63,52.77
2,Voluntary Non-Profit - Church,392,None,None,31.28,54.11
3,Proprietary,628,None,None,26.57,42.95
4,Government - State,56,None,None,30.57,73.56


In [23]:
"""============================================================================
5. Data quality audit
============================================================================"""
with open('sql/queries/05_data_completeness_by_topic.sql', 'r') as file:
    query = file.read()

df = pd.read_sql_query(query, conn)

df.head(5)

,topic,file_year,total_cells,reported,missing,pct_missing
0,Timely_and_Effective_Care,2026,276258,108197,168061,60.8
1,Complications_and_Deaths,2026,383120,205616,177504,46.3
2,HAIs,2026,172404,96356,76048,44.1
3,Readmissions_Reduction_Program,2026,91650,51439,40211,43.9
4,Payment_and_Value_of_Care,2026,4625,2886,1739,37.6


In [27]:
"""============================================================================
6. Medicare payment REDUCTION under the Hospital-Acquired Conditions program
   vs don't have
============================================================================"""
import pandas as pd

with open('sql/queries/06_hac_penalty_vs_infections.sql', 'r') as file:
    query = file.read()

df = pd.read_sql_query(query, conn)

df.head(5)

,payment_reduction,num_hospitals,avg_clabsi_sir,avg_cauti_sir,avg_mrsa_sir
0,No,2293,None,None,None
1,Yes,719,None,None,None


In [33]:
"""============================================================================
7.Data-quality QA check
============================================================================"""
with open('sql/queries/07_cross_file_consistency_check.sql', 'r') as file:
    query = file.read()

df = pd.read_sql_query(query, conn)

df.head(5)

,hospital_name,state,has_name_conflict_2026


In [36]:
"""============================================================================
8. Rank U.S. states by average AMI (heart attack)
============================================================================"""

with open('sql/queries/08_topic_coverage_by_state.sql', 'r') as file:
    query = file.read()

df = pd.read_sql_query(query, conn)

df.head(5)

,state,avg_topics_reported,num_hospitals
0,NJ,5.94,62
1,CT,5.61,28
2,SC,5.56,57
3,RI,5.55,11
4,MA,5.51,59
